Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs/001/L_Fore/01.bmp'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)


Preprocessing for Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
NUM_IMAGES = 9  # Protocol 3: first 9 images per finger

all_images = []
all_labels = []

# === STEP 1: LOAD INDIVIDUAL FINGER IMAGES (Strategy 2) ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Protocol 1 - Strategy 2"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_LIST:
        finger_path = os.path.join(subject_path, finger)
        for img_idx in range(1, NUM_IMAGES + 1):
            img_path = os.path.join(finger_path, f"{img_idx:02d}.bmp")
            print(f"📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                print(f"❌ Missing image: {img_path}")
                continue

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            all_images.append(img_norm)
            label = f"{subj}_{finger}_img{img_idx:02d}"
            all_labels.append(label)
            print(f"✅ Added sample: {label}")

all_labels = np.array(all_labels)
print(f"\n✅ Total Samples Loaded: {len(all_images)}")
print(f"✅ Example Image Shape: {all_images[0].shape}")

# === STEP 2: COMPUTE (2D)²PCA PROJECTION MATRICES (47×47) ===
def compute_2d2pca_projection(images, num_row_components, num_col_components):
    print("\n⚙️ Computing (2D)²PCA projection matrices...")
    n = len(images)
    h, w = images[0].shape
    mean_img = sum(images) / n

    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))

    for i, img in enumerate(images):
        A = img - mean_img
        G_row += A @ A.T
        G_col += A.T @ A
        if i < 3:
            print(f"  ➕ Sample {i+1} contribution added")

    G_row /= n
    G_col /= n

    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)

    idx_r = np.argsort(-eig_vals_r)
    idx_c = np.argsort(-eig_vals_c)

    U = eig_vecs_r[:, idx_r[:num_row_components]]
    V = eig_vecs_c[:, idx_c[:num_col_components]]

    print(f"✅ U (row) shape: {U.shape}, V (col) shape: {V.shape}")
    return U, V

# Use 47 × 47 components
num_row_components = 47
num_col_components = 47

U, V = compute_2d2pca_projection(all_images, num_row_components, num_col_components)

# === STEP 3: PROJECT IMAGES USING (2D)²PCA ===
projected_features = []
for i, img in enumerate(all_images):
    feat = U.T @ img @ V
    projected_features.append(feat)
    if i < 3:
        print(f"🧮 Projected shape of sample {i+1}: {feat.shape}")

# === STEP 4: FLATTEN FOR CLASSIFIER ===
flat_features = np.array([f.flatten() for f in projected_features])
print(f"\n✅ Final flattened feature matrix shape: {flat_features.shape}")
print(f"🧾 Number of labels: {len(all_labels)}")


Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
TEST_INDICES = [10]  # Protocol 3 test indices (images10)

test_images = []
test_labels = []
test_paths = []

# === LOAD TEST DATA (Strategy 2: individual finger samples) ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading Protocol 1 - Strategy 2 (Test Set)"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_LIST:
        finger_path = os.path.join(subject_path, finger)
        for img_idx in TEST_INDICES:
            img_path = os.path.join(finger_path, f"{img_idx:02d}.bmp")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"⚠️ Missing: {img_path}")
                continue

            print(f"✅ Using: {img_path}")
            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)

            test_images.append(img_norm)
            test_labels.append(f"{subj}_{finger}_img{img_idx:02d}")
            test_paths.append(img_path)

# === PROJECT TEST IMAGES USING (2D)²PCA ===
proj_test_features = [U.T @ img @ V for img in test_images]
flat_test_features = np.array([f.flatten() for f in proj_test_features])
test_labels = np.array(test_labels)

print(f"\n✅ Projected test features shape: {flat_test_features.shape}")
print(f"🧾 Number of test samples: {len(flat_test_features)}")
print(f"🧾 Example test label: {test_labels[0]}")


Benchmarking:

In [ ]:
# === CLASSIFICATION USING MANHATTAN DISTANCE (SUBJECT + FINGER MATCH) ===

correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_L_Fore_img08"

    # 📏 Compute Manhattan distances to all training samples
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # 🏆 Identify the nearest training sample
    min_index = np.argmin(distances)
    predicted_label = all_labels[min_index]  # e.g., "005_L_Fore_img03"

    print(f"\n🔹 Test sample {i+1}:")
    print(f"   🎯 Predicted → {predicted_label}")
    print(f"   ✅ Actual    → {true_label}")

    # Extract subject ID and finger from both labels
    pred_subject, pred_finger = predicted_label.split('_')[0], predicted_label.split('_')[1]
    true_subject, true_finger = true_label.split('_')[0], true_label.split('_')[1]

    # 🎯 Match check (subject + finger)
    if pred_subject == true_subject and pred_finger == true_finger:
        correct_matches += 1
        print("   🟢 Match (Subject + Finger correct)")
    else:
        print("   🔴 Mismatch")

# 📈 Compute and display final recognition accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final Recognition Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests})")
